In [ ]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

In [ ]:
import torch

model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import os
from PIL import Image
import pandas as pd


images_path = '/content/drive/MyDrive/yolo_project/dataset/valid/images'
labels_path = '/content/drive/MyDrive/yolo_project/dataset/valid/labels'

MAX_CAPACITY = 20

results_data = []

# ===== 4. функция чтения разметки =====
def get_true_count(label_file):
    if not os.path.exists(label_file):
        return 0

    with open(label_file, 'r') as f:
        lines = f.readlines()

    return len(lines)

# ===== 5. основной цикл =====
for img_name in os.listdir(images_path):
    if not img_name.endswith('.jpg'):
        continue

    img_path = os.path.join(images_path, img_name)
    label_path = os.path.join(labels_path, img_name.replace('.jpg', '.txt'))

    # предсказание
    results = model(img_path)

    pred_count = 0
    confidences = []

    for *box, conf, cls in results.xyxy[0]:
        if int(cls) == 0:  # person
            pred_count += 1
            confidences.append(float(conf))

    # истинное значение
    true_count = get_true_count(label_path)

    # метрики
    error = abs(pred_count - true_count)

    occupancy_pred = (pred_count / MAX_CAPACITY) * 100
    occupancy_true = (true_count / MAX_CAPACITY) * 100

    occupancy_error = abs(occupancy_pred - occupancy_true)

    avg_conf = sum(confidences)/len(confidences) if confidences else 0

    # сохранение
    results_data.append({
        'image': img_name,
        'true_count': true_count,
        'pred_count': pred_count,
        'error': error,
        'occupancy_true_%': occupancy_true,
        'occupancy_pred_%': occupancy_pred,
        'occupancy_error': occupancy_error,
        'avg_conf': avg_conf
    })

# ===== 6. в таблицу =====
df = pd.DataFrame(results_data)

# ===== 7. общая статистика =====
print('===== BASELINE METRICS =====')
print('Средняя ошибка (люди):', df['error'].mean())
print('Средняя ошибка заполненности (%):', df['occupancy_error'].mean())
print('Средняя уверенность:', df['avg_conf'].mean())

# ===== 8. сохранить =====
df.to_csv('/content/drive/MyDrive/yolo_project/baseline_results.csv', index=False)

print('Сохранено в baseline_results.csv')

In [ ]:
print("===== BASELINE SUMMARY =====")
print("Количество изображений:", len(df))
print("Среднее истинное количество людей:", round(df['true_count'].mean(), 2))
print("Среднее предсказанное количество людей:", round(df['pred_count'].mean(), 2))
print("Средняя ошибка подсчёта:", round(df['error'].mean(), 2))
print("Максимальная ошибка подсчёта:", df['error'].max())
print("Средняя ошибка заполненности (%):", round(df['occupancy_error'].mean(), 2))
print("Средняя уверенность модели:", round(df['avg_conf'].mean(), 3))



In [ ]:
%cd /content/yolov5

!WANDB_MODE=disabled python train.py \
  --img 640 \
  --batch 16 \
  --epochs 30 \
  --data /content/drive/MyDrive/yolo_project/dataset/data.yaml \
  --weights yolov5s.pt \
  --name metro_yolov5s_e10

In [ ]:
import torch
import os
from PIL import Image
import pandas as pd

model = torch.hub.load(
    'ultralytics/yolov5',
    'custom',
    path='/content/yolov5/runs/train/metro_yolov5s_e10/weights/best.pt'
)
images_path = '/content/drive/MyDrive/yolo_project/dataset/valid/images'
labels_path = '/content/drive/MyDrive/yolo_project/dataset/valid/labels'

MAX_CAPACITY = 20

results_data = []

# ===== 4. функция чтения разметки =====
def get_true_count(label_file):
    if not os.path.exists(label_file):
        return 0

    with open(label_file, 'r') as f:
        lines = f.readlines()

    return len(lines)

# ===== 5. основной цикл =====
for img_name in os.listdir(images_path):
    if not img_name.endswith('.jpg'):
        continue

    img_path = os.path.join(images_path, img_name)
    label_path = os.path.join(labels_path, img_name.replace('.jpg', '.txt'))

    # предсказание
    results = model(img_path)

    pred_count = 0
    confidences = []

    for *box, conf, cls in results.xyxy[0]:
        if int(cls) == 0:  # person
            pred_count += 1
            confidences.append(float(conf))

    # истинное значение
    true_count = get_true_count(label_path)

    # метрики
    error = abs(pred_count - true_count)

    occupancy_pred = (pred_count / MAX_CAPACITY) * 100
    occupancy_true = (true_count / MAX_CAPACITY) * 100

    occupancy_error = abs(occupancy_pred - occupancy_true)

    avg_conf = sum(confidences)/len(confidences) if confidences else 0

    # сохранение
    results_data.append({
        'image': img_name,
        'true_count': true_count,
        'pred_count': pred_count,
        'error': error,
        'occupancy_true_%': occupancy_true,
        'occupancy_pred_%': occupancy_pred,
        'occupancy_error': occupancy_error,
        'avg_conf': avg_conf
    })

# ===== 6. в таблицу =====
df = pd.DataFrame(results_data)

# ===== 7. общая статистика =====
print('===== 10EPOCH METRICS =====')
print('Средняя ошибка (люди):', df['error'].mean())
print('Средняя ошибка заполненности (%):', df['occupancy_error'].mean())
print('Средняя уверенность:', df['avg_conf'].mean())

# ===== 8. сохранить =====
df.to_csv('/content/drive/MyDrive/yolo_project/10epoch_results.csv', index=False)

print('Сохранено')

print("===== 10EPOCH SUMMARY =====")
print("Количество изображений:", len(df))
print("Среднее истинное количество людей:", round(df['true_count'].mean(), 2))
print("Среднее предсказанное количество людей:", round(df['pred_count'].mean(), 2))
print("Средняя ошибка подсчёта:", round(df['error'].mean(), 2))
print("Максимальная ошибка подсчёта:", df['error'].max())
print("Средняя ошибка заполненности (%):", round(df['occupancy_error'].mean(), 2))
print("Средняя уверенность модели:", round(df['avg_conf'].mean(), 3))


In [ ]:
e10_model = torch.hub.load(
    'ultralytics/yolov5',
    'custom',
    path='/content/yolov5/runs/train/metro_yolov5s_e10/weights/best.pt'
)

e10_model.conf = 0.5
e10_model.iou = 0.20
e10_model.max_det = 30

img_path = '/content/drive/MyDrive/yolo_project/dataset/valid/images/img38_jpg.rf.3670b1d713dff0c8610bffff2b9a4e23.jpg'  # подставь любое имя из valid
results = e10_model(img_path)
results.show()

In [ ]:
import os
import pandas as pd
import torch

images_path = '/content/drive/MyDrive/yolo_project/dataset/valid/images'
labels_path = '/content/drive/MyDrive/yolo_project/dataset/valid/labels'

MAX_CAPACITY = 20

def get_true_count(label_file):
    if not os.path.exists(label_file):
        return 0

    with open(label_file, 'r') as f:
        lines = [line for line in f.readlines() if line.strip()]

    return len(lines)


def evaluate_model(model):
    rows = []

    for img_name in os.listdir(images_path):
        if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        img_path = os.path.join(images_path, img_name)
        label_path = os.path.join(labels_path, os.path.splitext(img_name)[0] + '.txt')

        results = model(img_path)

        pred_count = 0
        confidences = []

        for *box, conf, cls in results.xyxy[0]:
            if int(cls) == 0:
                pred_count += 1
                confidences.append(float(conf))

        true_count = get_true_count(label_path)

        count_error = abs(pred_count - true_count)
        occupancy_true = (true_count / MAX_CAPACITY) * 100
        occupancy_pred = (pred_count / MAX_CAPACITY) * 100
        occupancy_error = abs(occupancy_pred - occupancy_true)

        avg_conf = sum(confidences) / len(confidences) if confidences else 0

        rows.append({
            'image': img_name,
            'true_count': true_count,
            'pred_count': pred_count,
            'count_error': count_error,
            'occupancy_true_%': occupancy_true,
            'occupancy_pred_%': occupancy_pred,
            'occupancy_error_%': occupancy_error,
            'avg_conf': avg_conf
        })

    df = pd.DataFrame(rows)

    return {
        'mean_true_count': round(df['true_count'].mean(), 2),
        'mean_pred_count': round(df['pred_count'].mean(), 2),
        'mean_count_error': round(df['count_error'].mean(), 2),
        'max_count_error': int(df['count_error'].max()),
        'mean_occupancy_error_%': round(df['occupancy_error_%'].mean(), 2),
        'mean_conf': round(df['avg_conf'].mean(), 3)
    }, df

In [ ]:
trained30_model = torch.hub.load(
    'ultralytics/yolov5',
    'custom',
    path='/content/yolov5/runs/train/metro_yolov5s_e10/weights/best.pt'
)

In [ ]:
conf_values = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
iou_values = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45]
max_det_values = [15, 20, 25, 30, 40]

search_results = []

for conf in conf_values:
    for iou in iou_values:
        for max_det in max_det_values:
            trained30_model.conf = conf
            trained30_model.iou = iou
            trained30_model.max_det = max_det

            summary, _ = evaluate_model(trained30_model)

            search_results.append({
                'conf': conf,
                'iou': iou,
                'max_det': max_det,
                **summary
            })

search_df = pd.DataFrame(search_results)

# сортируем по главной метрике — ошибке подсчёта
search_df_sorted = search_df.sort_values(
    by=['mean_count_error', 'mean_occupancy_error_%', 'max_count_error'],
    ascending=[True, True, True]
)

search_df_sorted.head(20)